<div style="text-align: center; font-weight: bold;">County‑Level Random Forest Model for Predicting Preventable Hospitalization Rates</div>

This model estimates preventable hospitalization rates at the county level using a Random Forest regression approach. Instead of predicting hospital‑level readmissions—where predictors do not vary across hospitals within the same county, this framework focuses on counties, where social, demographic, health behavior, and provider‑supply variables genuinely differ. Modeling at the county level produces a clearer, more stable signal and supports practical decisions about where primary care access, chronic disease management, and community health investments are most needed.

In [28]:
# Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestRegressor


In [ ]:
# Database connection
from sqlalchemy import create_engine
engine = create_engine("postgresql+psycopg2://<username>:<password>@<host>:<port>/<database>")

df = pd.read_sql("SELECT * FROM vw_hospital_readmission_analytics", engine)

In [30]:
# Initial data exploration
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18264 entries, 0 to 18263
Data columns (total 74 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   hospital_key                      18264 non-null  int64  
 1   facility_id                       18264 non-null  object 
 2   hospital_name                     18264 non-null  object 
 3   city                              18264 non-null  object 
 4   state                             18264 non-null  object 
 5   zip_code                          18264 non-null  object 
 6   hospital_type                     18264 non-null  object 
 7   hospital_ownership                18264 non-null  object 
 8   emergency_services                18264 non-null  bool   
 9   overall_star_rating               15198 non-null  float64
 10  readm_measure_count               18156 non-null  float64
 11  readm_measures_better             17742 non-null  float64
 12  read

Compute missingness for every column

In [31]:
missing = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_pct")
)

missing_over = missing[missing["missing_pct"] > 0.30]
missing_over


,missing_pct
pqi_pneumonia_75plus,1.000000
pqi_uti_75plus,1.000000
pqi_copd_75plus,1.000000
pqi_diabetes_75plus,1.000000
pqi_hypertension_75plus,1.000000
number_of_readmissions,0.556888
excess_readmission_ratio,0.350964
predicted_readmission_rate,0.350964
expected_readmission_rate,0.350964


Check the target variable’s completeness

In [32]:
df["preventable_hospitalization_rate"].isna().mean()

np.float64(0.011498028909329829)

In [33]:
# Modeling by measure code
df_model = df[df["preventable_hospitalization_rate"].notna()].copy()
df_model.shape

(18054, 74)

In [34]:
# Drop PQI columns. Those columns are empty.
pqi_cols = [c for c in df.columns if "pqi_" in c]
df_model = df_model.drop(columns=pqi_cols)

<div style="text-align: center; font-weight: bold;">
    Modeling Strategy
</div>

Model preventable hospitalization rates at the county level by aggregating all hospital level rows into a single record per county and year. Use county level social, demographic, health behavior, and provider supply variables as predictors, since these factors meaningfully vary across counties and directly influence avoidable hospitalizations. After aggregation, split the dataset into features and target, preprocess numeric variables with median imputation and categorical variables with target encoding, and train a Random Forest regressor. Evaluate performance using 5 fold cross validation to obtain stable R² and RMSE estimates. Finally, extract feature importance to identify which county level conditions most strongly drive preventable hospitalizations and use these insights to guide public health planning and resource allocation.

Collapse df_model to county‑level rows

In [ ]:
county_df = (
    df_model
    .groupby(["fips_st_cnty", "county_name", "state", "year"])
    .agg({
        "preventable_hospitalization_rate": "mean",
        "poverty_pct": "mean",
        "unemployment_pct": "mean",
        "uninsured_pct": "mean",
        "income_ratio": "mean",
        "broadband_access_pct": "mean",
        "svi_overall_rank": "mean",
        "smoking_pct": "mean",
        "obesity_pct": "mean",
        "diabetes_pct": "mean",
        "primary_care_physicians_rate": "mean",
        "mental_health_provider_rate": "mean",
        "hospital_beds_per_1000": "mean",
        "population": "mean"
    })
    .reset_index()
)

Define X and y using county_df

In [ ]:
y = county_df["preventable_hospitalization_rate"]
X = county_df.drop(columns=[
    "preventable_hospitalization_rate", # Target variable
    "fips_st_cnty",
    "county_name",
    "state",
    "year"
])


Build the preprocessing + Random Forest pipeline

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", TargetEncoder(), cat_cols)
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ))
])

Evaluate with 5‑fold cross‑validation

In [38]:
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
rmse = np.sqrt(-cross_val_score(model, X, y, cv=5, scoring="neg_mean_squared_error"))

print("R2:", r2.mean())
print("RMSE:", rmse.mean())

R2: 0.15954280293609685
RMSE: 873.5711330642595


Fit the model and extract feature importance

In [40]:
model.fit(X, y)

rf = model.named_steps["rf"]
importances = rf.feature_importances_

feature_importance = (
    pd.DataFrame({"feature": num_cols + cat_cols, "importance": importances})
    .sort_values("importance", ascending=False)
)
feature_importance.head(20)

,feature,importance
0,poverty_pct,0.169194
7,obesity_pct,0.141971
3,income_ratio,0.084732
5,svi_overall_rank,0.077520
6,smoking_pct,0.076682
12,population,0.072846
8,diabetes_pct,0.070263
1,unemployment_pct,0.059480
4,broadband_access_pct,0.054668
10,mental_health_provider_rate,0.050617


<div style="text-align: center; font-weight: bold;">County‑level modeling is appropriate and meaningful</div>

Aggregating hospital‑level data into county‑year records produces a more stable and interpretable modeling framework. County‑level predictors — socioeconomic conditions, health behaviors, and provider supply, vary meaningfully across counties and directly influence preventable hospitalization rates. This makes the county the correct unit of analysis for understanding avoidable hospitalizations.

<div style="text-align: center; font-weight: bold;">The Random Forest model captures a real but modest signal</div>

The model achieves:
    
    R² ≈ 0.16

    RMSE ≈ 874

This means the model explains about 16% of the variation in preventable hospitalization rates across counties.
For county‑level public‑health outcomes, which are noisy, multifactorial, and influenced by unobserved structural factors, this level of performance is typical. It indicates that the predictors contain meaningful information, but additional variables (e.g., chronic disease burden, outpatient utilization, Medicaid expansion status) would likely improve performance.

<div style="text-align: center; font-weight: bold;">Social determinants and health behaviors are the strongest drivers</div>

    Poverty is the strongest predictor, highlighting the central role of socioeconomic disadvantage in avoidable hospitalizations.

    Obesity, smoking, and diabetes, core chronic‑disease risk factors, are major contributors.

    Income inequality (income_ratio) and SVI rank reinforce the importance of structural vulnerability.

    Population size likely captures urban–rural differences and access patterns.

These results align with established public‑health research: preventable hospitalizations are driven by a combination of economic hardship, chronic disease burden, and limited access to preventive care.